In [42]:
import os
import gzip
import csv
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## download imdb gz file

In [43]:
import os
import gzip
import csv
import requests

BASE_URL = "https://datasets.imdbws.com/"
FILES = [
    "title.basics.tsv.gz",
    "title.ratings.tsv.gz",
    "name.basics.tsv.gz",
    "title.principals.tsv.gz",
    "title.crew.tsv.gz",
    "title.akas.tsv.gz",
    "title.episode.tsv.gz",
]

OUTPUT_DIR = "preview"
MAX_ROWS = 1_000  # 🔽 changed from 10k to 1k

os.makedirs(OUTPUT_DIR, exist_ok=True)

def download_file(filename):
    url = BASE_URL + filename
    print(f"Downloading {filename}...")
    r = requests.get(url, stream=True)
    r.raise_for_status()

    with open(filename, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

def extract_preview(gz_file):
    output_csv = os.path.join(
        OUTPUT_DIR, gz_file.replace(".tsv.gz", "_1k.csv")
    )

    print(f"Creating preview {output_csv}...")

    with gzip.open(gz_file, "rt", encoding="utf-8") as fin, \
         open(output_csv, "w", newline="", encoding="utf-8") as fout:

        reader = csv.reader(fin, delimiter="\t")
        writer = csv.writer(fout)

        for i, row in enumerate(reader):
            writer.writerow(row)
            if i + 1 >= MAX_ROWS:
                break

def main():
    for file in FILES:
        if not os.path.exists(file):
            download_file(file)
        extract_preview(file)

    print("✅ IMDb 1K previews created")

if __name__ == "__main__":
    main()


Creating preview preview/title.basics_1k.csv...
Creating preview preview/title.ratings_1k.csv...
Creating preview preview/name.basics_1k.csv...
Creating preview preview/title.principals_1k.csv...
Creating preview preview/title.crew_1k.csv...
Creating preview preview/title.akas_1k.csv...
Creating preview preview/title.episode_1k.csv...
✅ IMDb 1K previews created


## load wiki person data

In [44]:
# ============================================================================
# 1. LOAD PERSON DATA
# ============================================================================

import pandas as pd
from pathlib import Path

print("=" * 80)
print("LOADING PERSON DATA")
print("=" * 80)

person_file = Path(r'processed/person.csv')  # same folder as script

try:
    person_df = pd.read_csv(person_file)
    print(f"Loaded {person_file.name} ({len(person_df)} rows)")
except Exception as e:
    print(f"Error loading {person_file.name}: {e}")


LOADING PERSON DATA
Loaded person.csv (116245 rows)


## wiki relation data

## title.akas

In [45]:

############################################################
# Step 0: Filter title.akas for CA/US/XWW and language
############################################################

# Regions and languages we care about
regions_to_keep = ["CA", "US", "XWW"]
languages_to_keep = ["en"]

akas_chunks = []

for chunk in pd.read_csv(
    "title.akas.tsv.gz",
    sep="\t",
    compression="gzip",
    low_memory=False,
    chunksize=100_000
):
    # Replace IMDb missing values
    chunk.replace("\\N", pd.NA, inplace=True)

    # Language filter: en or NA
    language_ok = chunk["language"].isin(languages_to_keep) | chunk["language"].isna()

    # Region filter
    region_ok = chunk["region"].isin(regions_to_keep)

    # Apply combined filter
    chunk = chunk[region_ok & language_ok].copy()

    # Optional year conversion
    if "year" in chunk.columns:
        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")

    # Map movie import region
    # CA → CA movie
    # US + XWW → US movie
    chunk["movie_import_region"] = chunk["region"].map(
        {"CA": "CA", "US": "US", "XWW": "US","IN": "US"}
    )

    akas_chunks.append(chunk)

# Combine filtered chunks
title_akas_filtered = pd.concat(akas_chunks, ignore_index=True)

# Split by movie import region
imdb_ca_titles = title_akas_filtered[
    title_akas_filtered["movie_import_region"] == "CA"
].copy()

imdb_us_titles = title_akas_filtered[
    title_akas_filtered["movie_import_region"] == "US"
].copy()

print("IMDb Canadian titles:", imdb_ca_titles.shape[0])
print("IMDb US titles:", imdb_us_titles.shape[0])

# CA + US movie IDs as a set
ca_us_title_ids = set(pd.concat([imdb_ca_titles["titleId"], imdb_us_titles["titleId"]]).unique())




IMDb Canadian titles: 253584
IMDb US titles: 1898725


## Principal

### people we care about: wiki person > IMDb 

In [46]:

############################################################
# Step 1: Extract people we care about
############################################################

# Assuming merged_personal DataFrame exists with column "nconst"
# Convert to a set for fast lookup
people_nconsts = set(person_df["IMDb"])

############################################################
# Step 2: Filter title.principals for our people
############################################################

principals_chunks = []

for chunk in pd.read_csv(
    "title.principals.tsv.gz",
    sep="\t",
    compression="gzip",
    low_memory=False,
    chunksize=200_000
):
    chunk.replace("\\N", pd.NA, inplace=True)

    # Keep only rows where the person appears in our people list
    filtered = chunk[chunk["nconst"].isin(people_nconsts)].copy()
    principals_chunks.append(filtered)

principals_filtered = pd.concat(principals_chunks, ignore_index=True)
# Set of movie IDs that have at least one of our people as principal
valid_tconsts_from_people = set(principals_filtered["tconst"])

print("principals_filtered:", principals_filtered.shape[0])


principals_filtered: 11887607


In [47]:
principals_filtered.to_csv(
    r"processed/principals.csv",
    index=False
)

## ## Filter title.basics

### Filter title.basics > movie

In [48]:
############################################################
# Step 3: Filter title.basics
############################################################

filtered_chunks = []

for chunk in pd.read_csv(
    "title.basics.tsv.gz",
    sep="\t",
    compression="gzip",
    low_memory=False,
    chunksize=250_000
):
    chunk.replace("\\N", pd.NA, inplace=True)

    # Movie only
    is_movie = chunk["titleType"] == "movie"

    # Year filter
    start_year = pd.to_numeric(chunk["startYear"], errors="coerce")
    # year_ok = start_year.between(1900, 2026)

    # CA / US filter
    in_ca_us = chunk["tconst"].isin(ca_us_title_ids)

    # Movie has at least one principal in people
    has_person = chunk["tconst"].isin(valid_tconsts_from_people)

    # Combine all conditions
    keep_rows = is_movie  & in_ca_us & has_person

    filtered = chunk.loc[keep_rows].copy()
    filtered_chunks.append(filtered)

# Combine filtered title.basics
title_basics_filtered = pd.concat(filtered_chunks, ignore_index=True)



# Sanity check
print("Movies kept:", title_basics_filtered.shape[0])
print("Unique titles:", title_basics_filtered["tconst"].nunique())




Movies kept: 160441
Unique titles: 160441


In [49]:
title_basics_filtered.to_csv(
    r"processed/title_movie.csv",
    index=False
)

### Filter title.basics > tvSeries

In [50]:
############################################################
# Step 3: Filter title.basics
############################################################

filtered_chunks = []

for chunk in pd.read_csv(
    "title.basics.tsv.gz",
    sep="\t",
    compression="gzip",
    low_memory=False,
    chunksize=250_000
):
    chunk.replace("\\N", pd.NA, inplace=True)

    # Movie only
    is_movie = chunk["titleType"] == "tvSeries"

    # Year filter
    start_year = pd.to_numeric(chunk["startYear"], errors="coerce")
    # year_ok = start_year.between(1900, 2026)

    # CA / US filter
    in_ca_us = chunk["tconst"].isin(ca_us_title_ids)

    # Movie has at least one principal in people
    has_person = chunk["tconst"].isin(valid_tconsts_from_people)

    # Combine all conditions
    keep_rows = is_movie  & in_ca_us & has_person

    filtered = chunk.loc[keep_rows].copy()
    filtered_chunks.append(filtered)

# Combine filtered title.basics
title_basics_filtered = pd.concat(filtered_chunks, ignore_index=True)



# Sanity check
print("tvSeries kept:", title_basics_filtered.shape[0])
print("Unique titles:", title_basics_filtered["tconst"].nunique())




tvSeries kept: 51875
Unique titles: 51875


In [51]:
title_basics_filtered.to_csv(
    r"processed/title_tvSeries.csv",
    index=False
)

## Rating

In [61]:
import pandas as pd

############################################################
# Step 0: Load valid tconsts from movie + tvSeries
############################################################

movie_tconsts = pd.read_csv(
    "processed/title_movie.csv",
    usecols=["tconst"]
)["tconst"]

tvSeries_tconsts = pd.read_csv(
    "processed/title_tvSeries.csv",
    usecols=["tconst"]
)["tconst"]

valid_tconst_set = set(
    pd.concat([movie_tconsts, tvSeries_tconsts])
    .dropna()
)

print(f"Total valid tconsts (movie + tvSeries): {len(valid_tconst_set):,}")

############################################################
# Step 1: Filter title.ratings.tsv.gz
############################################################

filter_in_chunks = []
filter_out_chunks = []

total_rows = 0

for chunk in pd.read_csv(
    "title.ratings.tsv.gz",
    sep="\t",
    compression="gzip",
    low_memory=False,
    chunksize=250_000
):
    total_rows += len(chunk)

    chunk.replace("\\N", pd.NA, inplace=True)

    # ---- filter_in (valid foreign key)
    filter_in = chunk[chunk["tconst"].isin(valid_tconst_set)]

    # ---- filter_out (invalid or missing foreign key)
    filter_out = chunk[
        chunk["tconst"].isna() |
        (~chunk["tconst"].isin(valid_tconst_set))
    ]

    if not filter_in.empty:
        filter_in_chunks.append(filter_in)

    if not filter_out.empty:
        filter_out_chunks.append(filter_out)

############################################################
# Step 2: Combine
############################################################

title_ratings_filter_in = pd.concat(filter_in_chunks, ignore_index=True)
title_ratings_filter_out = pd.concat(filter_out_chunks, ignore_index=True)

############################################################
# Step 3: Validation (raw = in + out)
############################################################

print("\n===== VALIDATION =====")
print(f"Total raw rows: {total_rows:,}")
print(f"Filter IN rows: {len(title_ratings_filter_in):,}")
print(f"Filter OUT rows: {len(title_ratings_filter_out):,}")
print(f"Check raw = in + out ? "
      f"{total_rows == len(title_ratings_filter_in) + len(title_ratings_filter_out)}")

print("\n===== DETAILS =====")
print(f"Unique tconsts in filter_in: {title_ratings_filter_in['tconst'].nunique():,}")
print(f"Rows with missing tconst (in filter_out): "
      f"{title_ratings_filter_out['tconst'].isna().sum():,}")


Total valid tconsts (movie + tvSeries): 212,316

===== VALIDATION =====
Total raw rows: 1,627,720
Filter IN rows: 138,597
Filter OUT rows: 1,489,123
Check raw = in + out ? True

===== DETAILS =====
Unique tconsts in filter_in: 138,597
Rows with missing tconst (in filter_out): 0


In [62]:
title_ratings_filtered.to_csv(
    r"processed/title_ratings.csv",
    index=False
)